# Comparativa de clustering con CV

Objetivo: comparar los métodos de clustering usados en la unidad actual mediante validación cruzada por folds, sin holdout.

## 1) Librerías

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn scipy


In [1]:
# Usamos sklearn para replicar la lógica de los ejemplos anteriores, pero ahora en formato comparativo.
# En clustering no hay variable objetivo de entrenamiento, así que la CV se interpreta como una forma de evaluar estabilidad y calidad del agrupamiento en distintas particiones.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

sns.set_theme(style="whitegrid")


## 2) Cargar y preparar datos

In [2]:
iris_data = pd.read_csv("iris.csv")
iris_data["variety"] = iris_data["variety"].astype("category")
print(iris_data.head())
print(iris_data.info())


   sepal.length  sepal.width  petal.length  petal.width variety
0           5.1          3.5           1.4          0.2  Setosa
1           4.9          3.0           1.4          0.2  Setosa
2           4.7          3.2           1.3          0.2  Setosa
3           4.6          3.1           1.5          0.2  Setosa
4           5.0          3.6           1.4          0.2  Setosa
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   sepal.length  150 non-null    float64 
 1   sepal.width   150 non-null    float64 
 2   petal.length  150 non-null    float64 
 3   petal.width   150 non-null    float64 
 4   variety       150 non-null    category
dtypes: category(1), float64(4)
memory usage: 5.0 KB
None


In [3]:
# Igual que en la unidad, trabajamos solo con las columnas numéricas y las escalamos.
iris_numerico = iris_data.select_dtypes(include=["number"])
scaler = StandardScaler()
X = scaler.fit_transform(iris_numerico)
y_real = iris_data["variety"].cat.codes.to_numpy()


## 3) Definir métricas y grilla

In [4]:
# El alumno puede cambiar las métricas desde esta lista.
# Silhouette y Calinski-Harabasz: más alto es mejor.
# Davies-Bouldin: más bajo es mejor.
# ARI usa la etiqueta real solo como referencia externa para comparar qué tan cerca quedó el clustering de las especies.
metricas_disponibles = ["silhouette", "calinski_harabasz", "davies_bouldin", "adjusted_rand_index"]
metricas_seleccionadas = ["silhouette", "calinski_harabasz", "davies_bouldin"]

for metrica in metricas_seleccionadas:
    if metrica not in metricas_disponibles:
        raise ValueError(f"La métrica {metrica} no está disponible")

comparison_grid = [
    {"modelo": "KMeans", "version": "k=3", "estimador": KMeans(n_clusters=3, random_state=42, n_init=25)},
    {"modelo": "KMeans", "version": "k=4", "estimador": KMeans(n_clusters=4, random_state=42, n_init=25)},
    {"modelo": "Jerárquico", "version": "complete", "estimador": AgglomerativeClustering(n_clusters=3, linkage="complete", metric="euclidean")},
    {"modelo": "Jerárquico", "version": "average", "estimador": AgglomerativeClustering(n_clusters=3, linkage="average", metric="euclidean")},
    {"modelo": "Jerárquico", "version": "single", "estimador": AgglomerativeClustering(n_clusters=3, linkage="single", metric="euclidean")},
    {"modelo": "Jerárquico", "version": "ward", "estimador": AgglomerativeClustering(n_clusters=3, linkage="ward", metric="euclidean")},
]

pd.DataFrame([{"modelo": x["modelo"], "version": x["version"]} for x in comparison_grid])


,modelo,version
0,KMeans,k=3
1,KMeans,k=4
2,Jerárquico,complete
3,Jerárquico,average
4,Jerárquico,single
5,Jerárquico,ward


## 4) Validación cruzada

In [5]:
# En clustering, cada fold se agrupa por separado.
# No estamos prediciendo etiquetas futuras, sino observando si la calidad del agrupamiento se mantiene cuando cambia la muestra analizada.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

for item in comparison_grid:
    resultados_folds = []

    for fold_id, (_, idx_valid) in enumerate(cv.split(X), start=1):
        X_fold = X[idx_valid]
        y_fold = y_real[idx_valid]

        labels = item["estimador"].fit_predict(X_fold)
        n_clusters_fold = len(np.unique(labels))

        fila_fold = {"fold": fold_id}

        if n_clusters_fold > 1 and "silhouette" in metricas_seleccionadas:
            fila_fold["silhouette"] = silhouette_score(X_fold, labels)
        if n_clusters_fold > 1 and "calinski_harabasz" in metricas_seleccionadas:
            fila_fold["calinski_harabasz"] = calinski_harabasz_score(X_fold, labels)
        if n_clusters_fold > 1 and "davies_bouldin" in metricas_seleccionadas:
            fila_fold["davies_bouldin"] = davies_bouldin_score(X_fold, labels)
        if "adjusted_rand_index" in metricas_seleccionadas:
            fila_fold["adjusted_rand_index"] = adjusted_rand_score(y_fold, labels)

        resultados_folds.append(fila_fold)

    tabla_folds = pd.DataFrame(resultados_folds)
    fila = {"modelo": item["modelo"], "version": item["version"]}
    for metrica in metricas_seleccionadas:
        fila[f"cv_mean_{metrica}"] = tabla_folds[metrica].mean()
        fila[f"cv_std_{metrica}"] = tabla_folds[metrica].std()
    resultados.append(fila)

resultados_df = pd.DataFrame(resultados)
resultados_df


,modelo,version,cv_mean_silhouette,cv_std_silhouette,cv_mean_calinski_harabasz,cv_std_calinski_harabasz,cv_mean_davies_bouldin,cv_std_davies_bouldin
0,KMeans,k=3,0.474323,0.034494,50.046679,12.091578,0.758790,0.065768
1,KMeans,k=4,0.475034,0.037107,47.869282,7.574490,0.672111,0.076328
2,Jerárquico,complete,0.498256,0.043269,46.617001,15.196444,0.696123,0.148974
3,Jerárquico,average,0.520978,0.030802,42.878538,15.267641,0.547290,0.110561
4,Jerárquico,single,0.511432,0.050682,38.881674,10.021890,0.539246,0.105911
5,Jerárquico,ward,0.480908,0.061101,47.059420,13.662534,0.707885,0.060680


## 5) Comparar resultados

In [6]:
# Ordenamos por la primera métrica seleccionada.
# Si esa métrica es Davies-Bouldin, el mejor valor es el más bajo.
metrica_principal = metricas_seleccionadas[0]
ascending = metrica_principal == "davies_bouldin"
columnas = ["modelo", "version"]
for metrica in metricas_seleccionadas:
    columnas.extend([f"cv_mean_{metrica}", f"cv_std_{metrica}"])

comparacion = resultados_df[columnas].sort_values(by=f"cv_mean_{metrica_principal}", ascending=ascending)
comparacion


,modelo,version,cv_mean_silhouette,cv_std_silhouette,cv_mean_calinski_harabasz,cv_std_calinski_harabasz,cv_mean_davies_bouldin,cv_std_davies_bouldin
3,Jerárquico,average,0.520978,0.030802,42.878538,15.267641,0.547290,0.110561
4,Jerárquico,single,0.511432,0.050682,38.881674,10.021890,0.539246,0.105911
2,Jerárquico,complete,0.498256,0.043269,46.617001,15.196444,0.696123,0.148974
5,Jerárquico,ward,0.480908,0.061101,47.059420,13.662534,0.707885,0.060680
1,KMeans,k=4,0.475034,0.037107,47.869282,7.574490,0.672111,0.076328
0,KMeans,k=3,0.474323,0.034494,50.046679,12.091578,0.758790,0.065768
